In [0]:
# Importar las librerías necesarias
import warnings
warnings.filterwarnings("ignore", message=".*threadpoolctl.*")
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

# Modelos que vamos a probar
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# --- Carga de Datos ---
# Cargamos la tabla que preparamos para el modelo de clasificación
print("Cargando datos desde Spark...")
dataset_spark = spark.table("workspace.data.pedidos_distribucion_target")

# Convertimos a Pandas
print("Convirtiendo a Pandas DataFrame...")
pandas_df = dataset_spark.toPandas()

# Separamos las variables predictoras (X) de la variable objetivo (y)
X = pandas_df.drop(["cliente_id", "target_prox_trx_digital"], axis=1)
y = pandas_df["target_prox_trx_digital"]

# Dividimos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Datos listos para modelar.")
print(f"Tamaño del set de entrenamiento: {len(X_train)} filas")
print(f"Tamaño del set de prueba: {len(X_test)} filas")

In [0]:
# Identificar columnas por tipo
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64', 'double']).columns

# Crear el transformador de preprocesamiento
# - A las variables numéricas les aplicaremos un escalado estándar.
# - A las variables categóricas les aplicaremos One-Hot Encoding.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [0]:
# Definir los modelos que queremos probar
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Configurar el experimento en MLflow
mlflow.set_experiment("/Users/nicolascristanchomurcia@gmail.com/pedidos_distribucion_target_sklearn")

for model_name, model in models.items():
    print(f"--- Entrenando modelo: {model_name} ---")
    
    with mlflow.start_run(run_name=model_name):
        
        pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', model)])
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  ROC AUC: {roc_auc:.4f}")
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)
        
        # --- MEJORA APLICADA AQUÍ ---
        # 1. Creamos un ejemplo de entrada con las primeras 5 filas del set de entrenamiento
        input_example = X_train.head(5)
        
        # 2. Añadimos el ejemplo al registrar el modelo
        mlflow.sklearn.log_model(
            sk_model=pipeline, 
            artifact_path="sklearn-model",
            input_example=input_example
        )

print("--- Proceso completado ---")

In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# --- 1. Definir los Modelos y la Grilla de Parámetros ---

# Para RandomForest, probaremos 4x3x2 = 24 combinaciones
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 150, 200],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_leaf': [1, 2]
}

# Para GradientBoosting, probaremos 3x3x3 = 27 combinaciones
param_grid_gb = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}

models_to_tune = {
    "RandomForest": (RandomForestClassifier(random_state=42), param_grid_rf),
    "GradientBoosting": (GradientBoostingClassifier(random_state=42), param_grid_gb)
}

# --- 2. Configurar el Experimento y Autologging ---
mlflow.set_experiment("/Users/nicolascristanchomurcia@gmail.com/pedidos_distribucion_target_sklearn")

# ¡La magia de autolog! Hará el registro de cada modelo por nosotros.
mlflow.sklearn.autolog()

# --- 3. Iterar y Entrenar con GridSearchCV ---
for model_name, (model, param_grid) in models_to_tune.items():
    print(f"--- Iniciando Grid Search para: {model_name} ---")
    
    # Crear el pipeline completo
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', model)])
    
    # Configurar GridSearchCV
    # cv=3 significa validación cruzada de 3 pliegues.
    # scoring='roc_auc' es la métrica que usaremos para decidir qué modelo es el mejor.
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
    
    # Ejecutar la búsqueda. Esto entrenará todos los modelos.
    grid_search.fit(X_train, y_train)
    
    # MLflow autologging registrará todo automáticamente.
    # El run principal contendrá el resumen y el mejor modelo.
    # Cada combinación de parámetros será un "run" anidado.
    
print("--- Proceso de Grid Search completado ---")

# Desactivar autologging al final (buena práctica)
mlflow.sklearn.autolog(disable=True)